Each week, you will apply the concepts of that week to your Integrated Capstone Project’s dataset. In preparation for Milestone One, create a Jupyter Notebook (similar to in Module B, semester two) that illustrates these lessons. There are no specific questions to answer in your Jupyter Notebook files in this course; your general goal is to analyze your data, using the methods you have learned about in this course and in this program, and draw interesting conclusions. 

For Week 2, include concepts such as linear regression with lasso, ridge, and elastic net regression. This homework will be submitted for peer review and feedback in Week 3 in the assignment titled 3.4 Peer Review: Week 2 Jupyter Notebook. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday.

In Week 7, you will compile your findings from your Jupyter Notebook homework into your Milestone One assignment for grading. For full instructions and the rubric for Milestone One, refer to the following link. 

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/jamieconner/projects/bu/dx799_01/mod_c_capstone


In [13]:
from src.load_data import load_wisconsin, load_brazil, split_X_y

df_brazil, target = load_brazil(sample_size= 50_000)
X_brazil, y_brazil = split_X_y(df_brazil, target)


In [14]:


X_brazil.info()
y_brazil

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 700187 to 754772
Data columns (total 39 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   age                                        50000 non-null  float64
 1   profession_code                            50000 non-null  float64
 2   morphology_code                            50000 non-null  int64  
 3   gender_female                              50000 non-null  int64  
 4   gender_unknown                             50000 non-null  int64  
 5   gender_male                                50000 non-null  int64  
 6   race_color_yellow                          50000 non-null  int64  
 7   race_color_white                           50000 non-null  int64  
 8   race_color_indigenous                      50000 non-null  int64  
 9   race_color_brown                           50000 non-null  int64  
 10  race_color_black     

700187     0
1473092    0
1119056    0
1614127    0
1714606    0
          ..
603780     1
278573     0
768371     0
558743     0
754772     0
Name: deceased, Length: 50000, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
import pandas as pd


X_train, X_test, y_train, y_test = train_test_split(
    X_brazil,
    y_brazil, 
    random_state=42, 
    test_size=.2, 
    stratify = y_brazil
    )

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, solver="saga", class_weight="balanced"))
])


grid_params = [
    {"model__penalty":["l2"], "model__C": [0.01, 0.1, 1, 10]},
    {"model__penalty":["l1"], "model__C": [0.01, 0.1, 1, 10]},
    
    {
      "model__penalty":["elasticnet"], 
      "model__C": [0.01, 0.1, 1, 10],
      "model__l1_ratio": [0.2, 0.5, 0.8]
    }
    
]

grid = GridSearchCV(
    estimator=pipe,
    param_grid= grid_params,
    cv = 5,
    scoring = "roc_auc",
    n_jobs= -1,
    verbose=1
)

grid.fit(X_train, y_train)

results= pd.DataFrame(grid.cv_results_)
best_model = grid.best_estimator_
best_params = grid.best_params_
best_cv_score = grid.best_score_



print(f"best model: {best_model}")
print(f"best params:{best_params}")
print(f"best_cv_score: {best_cv_score}")

results.head()




Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/jamieconner/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jamieconner/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jamieconner/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jamieconner/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


best model: Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=3000, penalty='l1',
                                    solver='saga'))])
best params:{'model__C': 0.1, 'model__penalty': 'l1'}
best_cv_score: 0.9388596018984202


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__penalty,param_model__l1_ratio,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,4.524665,5.908927,0.008317,0.001980,0.01,l2,NaN,"{'model__C': 0.01, 'model__penalty': 'l2'}",0.944551,0.935797,0.937188,0.936941,0.938796,0.938655,0.003100,20
1,5.100587,2.334826,0.008202,0.002480,0.10,l2,NaN,"{'model__C': 0.1, 'model__penalty': 'l2'}",0.944541,0.936274,0.937474,0.936997,0.938834,0.938824,0.002978,3
2,14.429821,5.869457,0.007197,0.002039,1.00,l2,NaN,"{'model__C': 1, 'model__penalty': 'l2'}",0.944542,0.935729,0.937454,0.936945,0.938810,0.938696,0.003085,18
3,20.689838,4.520916,0.004455,0.002047,10.00,l2,NaN,"{'model__C': 10, 'model__penalty': 'l2'}",0.944538,0.935796,0.937457,0.936939,0.938816,0.938709,0.003072,16
4,23.732646,25.014743,0.003677,0.001021,0.01,l1,NaN,"{'model__C': 0.01, 'model__penalty': 'l1'}",0.944856,0.935890,0.937078,0.936533,0.939196,0.938711,0.003267,15


In [21]:
results.sort_values("rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__penalty,param_model__l1_ratio,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
5,18.500461,19.790557,0.006477,0.002387,0.10,l1,NaN,"{'model__C': 0.1, 'model__penalty': 'l1'}",0.944583,0.936322,0.937601,0.936885,0.938908,0.938860,0.002989,1
12,7.829585,2.708322,0.004536,0.001821,0.10,elasticnet,0.5,"{'model__C': 0.1, 'model__l1_ratio': 0.5, 'mod...",0.944539,0.936312,0.937503,0.936952,0.938878,0.938837,0.002974,2
1,5.100587,2.334826,0.008202,0.002480,0.10,l2,NaN,"{'model__C': 0.1, 'model__penalty': 'l2'}",0.944541,0.936274,0.937474,0.936997,0.938834,0.938824,0.002978,3
6,21.413384,3.002308,0.004071,0.001439,1.00,l1,NaN,"{'model__C': 1, 'model__penalty': 'l1'}",0.944564,0.936164,0.937466,0.936931,0.938818,0.938789,0.003015,4
11,6.071864,1.333882,0.003838,0.001480,0.10,elasticnet,0.2,"{'model__C': 0.1, 'model__l1_ratio': 0.2, 'mod...",0.944578,0.936198,0.937337,0.936977,0.938844,0.938787,0.003020,5
8,2.055539,0.839998,0.004147,0.001570,0.01,elasticnet,0.2,"{'model__C': 0.01, 'model__l1_ratio': 0.2, 'mo...",0.944659,0.936074,0.937309,0.936889,0.938967,0.938780,0.003087,6
10,2.476906,0.758305,0.005910,0.002028,0.01,elasticnet,0.8,"{'model__C': 0.01, 'model__l1_ratio': 0.8, 'mo...",0.944842,0.935871,0.937339,0.936620,0.939200,0.938774,0.003229,7
16,25.685741,14.902311,0.005409,0.002895,1.00,elasticnet,0.8,"{'model__C': 1, 'model__l1_ratio': 0.8, 'model...",0.944587,0.936114,0.937386,0.936935,0.938819,0.938768,0.003039,8
9,2.369789,1.029552,0.005402,0.003029,0.01,elasticnet,0.5,"{'model__C': 0.01, 'model__l1_ratio': 0.5, 'mo...",0.944734,0.935837,0.937298,0.936727,0.939173,0.938754,0.003183,9
15,18.506825,4.796021,0.006550,0.002716,1.00,elasticnet,0.5,"{'model__C': 1, 'model__l1_ratio': 0.5, 'model...",0.944553,0.935915,0.937520,0.936940,0.938809,0.938747,0.003050,10


For Week 2, I compared ridge, lasso, and elastic net regularized logistic regression on the Brazil dataset using cross-validated ROC-AUC as the primary metric. Because the dataset contains over 1.7 million rows, I used a random sample to keep the grid search computationally manageable. I also used a stratified train/test split because the target variable was imbalanced. The best cross-validated model was lasso logistic regression with C = 0.1, which indicates relatively strong regularization and suggests that a sparser model generalized slightly better than the alternatives.

However, the performance differences across ridge, lasso, and elastic net were small, with mean CV scores within roughly a couple thousandths of one another. This suggests that the choice of regularization type did not dramatically change predictive performance on this dataset. Even so, lasso was still the most useful result because it provided the best CV score while also encouraging a simpler model through coefficient shrinkage and feature selection. I also tested class_weight="balanced" because of the class imbalance, but it did not materially improve ROC-AUC, suggesting that class weighting was not a major driver of performance for this particular modeling objective. 

In [ ]:
df_wisc, target = load_wisconsin()
X_wisc, y_wisc = split_X_y(df_wisc, target)

In [26]:

X_train, X_test, y_train, y_test = train_test_split(
    X_wisc,
    y_wisc, 
    random_state=42, 
    test_size=.2, 
    stratify = y_wisc
    )

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10_000, solver="saga"))
])


grid_params = [
    {"model__penalty":["l2"], "model__C": [0.01, 0.1, 1, 10]},
    {"model__penalty":["l1"], "model__C": [0.01, 0.1, 1, 10]},
    
    {
      "model__penalty":["elasticnet"], 
      "model__C": [0.01, 0.1, 1, 10],
      "model__l1_ratio": [0.2, 0.5, 0.8]
    }
    
]

grid = GridSearchCV(
    estimator=pipe,
    param_grid= grid_params,
    cv = 5,
    scoring = "roc_auc",
    n_jobs= -1,
    verbose=1
)

grid.fit(X_train, y_train)

results= pd.DataFrame(grid.cv_results_)
best_model = grid.best_estimator_
best_params = grid.best_params_
best_cv_score = grid.best_score_



print(f"best model: {best_model}")
print(f"best params:{best_params}")
print(f"best_cv_score: {best_cv_score}")

sorted_results = results.sort_values("rank_test_score")
sorted_results



Fitting 5 folds for each of 20 candidates, totalling 100 fits
best model: Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=10, max_iter=10000, solver='saga'))])
best params:{'model__C': 10, 'model__penalty': 'l2'}
best_cv_score: 0.9959752321981424


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__penalty,param_model__l1_ratio,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
3,0.143572,0.017519,0.002888,0.001210,10.00,l2,NaN,"{'model__C': 10, 'model__penalty': 'l2'}",0.997936,1.000000,0.987616,1.000000,0.994324,0.995975,0.004666,1
19,0.276256,0.030045,0.001298,0.000148,10.00,elasticnet,0.8,"{'model__C': 10, 'model__l1_ratio': 0.8, 'mode...",0.997936,1.000000,0.987616,1.000000,0.993808,0.995872,0.004707,2
17,0.225065,0.015981,0.001265,0.000107,10.00,elasticnet,0.2,"{'model__C': 10, 'model__l1_ratio': 0.2, 'mode...",0.997936,1.000000,0.987616,1.000000,0.993808,0.995872,0.004707,2
18,0.273315,0.050985,0.002361,0.001798,10.00,elasticnet,0.5,"{'model__C': 10, 'model__l1_ratio': 0.5, 'mode...",0.997420,1.000000,0.987616,1.000000,0.993808,0.995769,0.004666,4
7,0.403435,0.063522,0.001777,0.000558,10.00,l1,NaN,"{'model__C': 10, 'model__penalty': 'l1'}",0.997420,1.000000,0.987100,1.000000,0.994324,0.995769,0.004812,5
2,0.030082,0.005285,0.002387,0.000742,1.00,l2,NaN,"{'model__C': 1, 'model__penalty': 'l2'}",0.998452,1.000000,0.984004,0.998968,0.994324,0.995150,0.005898,6
14,0.062073,0.005932,0.001864,0.000738,1.00,elasticnet,0.2,"{'model__C': 1, 'model__l1_ratio': 0.2, 'model...",0.998452,1.000000,0.984004,0.998968,0.994324,0.995150,0.005898,6
6,0.286708,0.095063,0.002280,0.000815,1.00,l1,NaN,"{'model__C': 1, 'model__penalty': 'l1'}",0.998452,1.000000,0.984004,0.998452,0.993292,0.994840,0.005874,8
15,0.089328,0.009276,0.002281,0.000483,1.00,elasticnet,0.5,"{'model__C': 1, 'model__l1_ratio': 0.5, 'model...",0.998452,1.000000,0.982456,0.998968,0.993808,0.994737,0.006497,9
16,0.140562,0.028978,0.001866,0.000610,1.00,elasticnet,0.8,"{'model__C': 1, 'model__l1_ratio': 0.8, 'model...",0.998452,1.000000,0.982456,0.998452,0.993808,0.994634,0.006433,10


Compared with the unregularized or baseline logistic model from Week 1, regularized logistic regression produced very similar performance. The best Week 2 model used L2 regularization with C=10, but several L1 and elastic net models had nearly identical cross-validation scores. This suggests that the Wisconsin data are already clean and strongly separable, so regularization mainly provides stability rather than a major performance improvement. The main value of regularization here is not boosting accuracy, but controlling coefficient size and reducing risk from correlated tumor-measurement features.